# Установка

In [1]:
%pip install pymorphy2 catboost
%pip install deeppavlov==1.7.0
%pip install transformers==4.30.0 huggingface_hub==0.36.2 tokenizers==0.13.2
%pip install protobuf==3.20.0 pytorch-crf==0.7.2 sentencepiece==0.2.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#  Загрузка данных

In [2]:
import pandas as pd

df = pd.read_csv('women-clothing-accessories.3-class.balanced.csv', sep='\t')

print(df.shape)
print(df.columns.tolist())
print(df['sentiment'].value_counts())
df.head()

(90000, 2)
['review', 'sentiment']
negative    30000
neautral    30000
positive    30000
Name: sentiment, dtype: int64


,review,sentiment
0,качество плохое пошив ужасный (горловина напер...,negative
1,"Товар отдали другому человеку, я не получила п...",negative
2,"Ужасная синтетика! Тонкая, ничего общего с пре...",negative
3,"товар не пришел, продавец продлил защиту без м...",negative
4,"Кофточка голая синтетика, носить не возможно.",negative


# Лемматизация

In [3]:
import pymorphy2

morph = pymorphy2.MorphAnalyzer()

def preprocess(text):
    tokens = str(text).lower().split()
    tokens = [morph.parse(t)[0].normal_form for t in tokens if t.isalpha()]
    return " ".join(tokens)

df['clean_text'] = df['review'].apply(preprocess)

print("До:  ", df['review'][0][:100])
print("После:", df['clean_text'][0][:100])

До:   качество плохое пошив ужасный (горловина наперекос) Фото не соответствует Ткань ужасная рисунок блек
После: качество плохой пошив ужасный фото не соответствовать ткань ужасный рисунок блёклый маленький рукав 


# TF-IDF + разбивка

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

print("Train:", X_train_vec.shape)
print("Test: ", X_test_vec.shape)

Train: (72000, 20000)
Test:  (18000, 20000)


# Три модели машинного обучения - логистическая регрессия, рандом форест и кэтбуст

регрессию и рандом форест взял как базовые, которые первые на ум приишли, а кэтбуст решил взять, так как она в статье использовалась

## Модель 1 - Логистическая регрессия

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_vec, y_train)
preds_lr = lr.predict(X_test_vec)

print("=== Логистическая регрессия ===")
print(classification_report(y_test, preds_lr))

=== Логистическая регрессия ===
              precision    recall  f1-score   support

    neautral       0.60      0.63      0.61      6060
    negative       0.71      0.69      0.70      5942
    positive       0.83      0.82      0.82      5998

    accuracy                           0.71     18000
   macro avg       0.71      0.71      0.71     18000
weighted avg       0.71      0.71      0.71     18000



## Модель 2 — Random Forest

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_vec, y_train)
preds_rf = rf.predict(X_test_vec)

print("=== Random Forest ===")
print(classification_report(y_test, preds_rf))

=== Random Forest ===
              precision    recall  f1-score   support

    neautral       0.58      0.60      0.59      6060
    negative       0.71      0.65      0.68      5942
    positive       0.77      0.81      0.79      5998

    accuracy                           0.68     18000
   macro avg       0.69      0.69      0.68     18000
weighted avg       0.69      0.68      0.68     18000



## Модель 3 — CatBoost

In [7]:
from catboost import CatBoostClassifier

cb = CatBoostClassifier(iterations=300, verbose=50, random_seed=42)
cb.fit(X_train_vec, y_train)
preds_cb = cb.predict(X_test_vec)

print("=== CatBoost ===")
print(classification_report(y_test, preds_cb))

Learning rate set to 0.272195
0:	learn: 1.0331028	total: 1.42s	remaining: 7m 5s
50:	learn: 0.7731389	total: 24.2s	remaining: 1m 57s
100:	learn: 0.7242036	total: 46s	remaining: 1m 30s
150:	learn: 0.6995979	total: 1m 7s	remaining: 1m 6s
200:	learn: 0.6849781	total: 1m 29s	remaining: 43.8s
250:	learn: 0.6739477	total: 1m 50s	remaining: 21.6s
299:	learn: 0.6667080	total: 2m 11s	remaining: 0us
=== CatBoost ===
              precision    recall  f1-score   support

    neautral       0.59      0.60      0.60      6060
    negative       0.72      0.66      0.69      5942
    positive       0.77      0.81      0.79      5998

    accuracy                           0.69     18000
   macro avg       0.69      0.69      0.69     18000
weighted avg       0.69      0.69      0.69     18000



# НЛП модели


как нлп модели взял DeepPavlov и Руберт 

## DeepPavlov

загрузка и проверка

In [8]:
from deeppavlov import build_model, configs

dp_model = build_model(configs.classifiers.rusentiment_bert, download=True)
print("Модель загружена")

2026-04-27 21:25:03.126 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/v1/classifiers/rusentiment_bert/rusentiment_bert_torch.tar.gz download because of matching hashes
e:\labs\kafedra\.venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\labs\kafedra\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForSequenceClassification: ['cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relatio

Модель загружена


In [9]:
# Берём 500 строк
sample = df.sample(500, random_state=42)
preds_dp = dp_model(sample['clean_text'].tolist())

# В rusentiment метки: positive, negative, neutral, speech, skip
# Маппим к нашим трём классам
def map_label(label):
    label = label.lower()
    if 'positive' in label: return 'positive'
    if 'negative' in label: return 'negative'
    return 'neutral'

preds_dp_mapped = [map_label(p) for p in preds_dp]

print("=== DeepPavlov (rusentiment_bert) ===")
print(classification_report(sample['sentiment'], preds_dp_mapped))

=== DeepPavlov (rusentiment_bert) ===
              precision    recall  f1-score   support

    neautral       0.00      0.00      0.00       173
    negative       0.52      0.40      0.45       168
     neutral       0.00      0.00      0.00         0
    positive       0.68      0.38      0.49       159

    accuracy                           0.25       500
   macro avg       0.30      0.19      0.23       500
weighted avg       0.39      0.25      0.31       500



e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.

## Руберт

загрузка и проверка

In [ ]:
from transformers import pipeline

ru_sentiment = pipeline(
    "text-classification",
    model="seara/rubert-tiny2-russian-sentiment",
    truncation=True,
    max_length=512
)

def hf_sentiment(text):
    result = ru_sentiment(str(text)[:512])[0]['label'].lower()
    # модель возвращает: positive, negative, neutral
    return result

# Та же выборка 500 строк
preds_hf = [hf_sentiment(t) for t in sample['review'].tolist()]

print("=== RuBERT Sentiment (HuggingFace) ===")
print(classification_report(sample['sentiment'], preds_hf))

e:\labs\kafedra\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
e:\labs\kafedra\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alex\.cache\huggingface\hub\models--seara--rubert-tiny2-russian-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. I

=== RuBERT Sentiment (HuggingFace) ===
              precision    recall  f1-score   support

    neautral       0.00      0.00      0.00       173
    negative       0.76      0.67      0.71       168
     neutral       0.00      0.00      0.00         0
    positive       0.89      0.86      0.87       159

    accuracy                           0.50       500
   macro avg       0.41      0.38      0.40       500
weighted avg       0.54      0.50      0.52       500



e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
e:\labs\kafedra\.venv\lib\site-packages\sklearn\metrics\_classification.

# Сравнение моделей

In [ ]:
from sklearn.metrics import f1_score

results = {
    'Логистическая регрессия': f1_score(y_test, preds_lr, average='weighted'),
    'Random Forest':           f1_score(y_test, preds_rf, average='weighted'),
    'CatBoost':                f1_score(y_test, preds_cb, average='weighted'),
    'DeepPavlov BERT':         f1_score(sample['sentiment'], preds_dp_mapped, average='weighted'),
    'RuBERT Sentiment':        f1_score(sample['sentiment'], preds_hf, average='weighted'),
}

comparison = pd.DataFrame.from_dict(results, orient='index', columns=['F1 (weighted)'])
comparison = comparison.sort_values('F1 (weighted)', ascending=False)
print(comparison) 

                         F1 (weighted)
Логистическая регрессия       0.712340
CatBoost                      0.691684
Random Forest                 0.684448
RuBERT Sentiment              0.515408
DeepPavlov BERT               0.306602


# Вывод

классические ML-модели с TF-IDF могут превосходить готовые NLP-модели, если те обучены на данных из другой предметной области.
Возможно (скорее всего) нлп модели показали бы лучший результат, если бы их тоже дополнительно обучали, но рассматриваются готовые чекпоинты